# Opus-distillation training -- qwen3.5-4b (Colab)

Trains the replacement cheap-tier extraction model from the Claude-Opus-teacher
dataset built by `scripts/build_extraction_dataset.py --from-capture`. Spec:
`docs/superpowers/specs/2026-10-17-opus-distillation-design.md` section 7-8. Runbook:
`docs/superpowers/distill-runbook.md` (this notebook is one step of it).

**This repo never rents a GPU and never runs this notebook.** You run it yourself on
Colab; everything below is self-contained -- it installs its own packages, takes the
HF token from Colab secrets or an env var, and never hardcodes one.

**Pass criteria (spec section 8, checked AFTER this notebook, outside Colab):**
validity >= 99% on held-out prompts (`scripts/eval_finetune.py`), and at least as good
as solar-pro4 on every `scripts/extraction_proof.py` metric.

**Runtime:** A100 (40GB) recommended for the fastest run at `max_seq_length=20480`;
an L4 (22.5GB) works with the smaller batch size noted below. Both configs use 4-bit
loading + gradient checkpointing -- see the "what fits" note in the training cell.

## 0. Design decisions carried over from the dataset builder

- **Loss weighting -> upsampling, not a custom weighted trainer.**
  `build_extraction_dataset.py`'s `--upsample` already repeats TRAIN records so
  `TARGET_MIX` (the per-prompt-type share: `extract_edges.edge` 34%,
  `dedupe_edges.resolve_edge` 25%, `extract_nodes.extract_summaries_batch` 18%,
  `extract_nodes.extract_text` 13%, `dedupe_nodes.nodes` 10%) is approached in TOKEN
  share (see `upsample_train`'s docstring). TRL's `SFTTrainer` (which Unsloth wraps)
  has no first-class notion of a per-example loss weight it would actually honour, so
  this notebook does NOT implement a custom weighted-loss `Trainer` subclass -- it
  trains plain SFT on the **upsampled** train split and relies on repetition to do the
  weighting. `meta.loss_weight` is still read below (every record carries it, and
  `meta.repeat_index` marks each repeated copy) purely to REPORT the realised mix as a
  sanity check, not as a literal per-example multiplier.
- **Completion-only loss masking -> build with `--loss-basis completion`.**
  This notebook trains with the prompt tokens masked out of the loss (see section 3).
  `assign_loss_weights`'s docstring is explicit that the weight VALUES depend on which
  masking regime they were computed for: `basis="full"` (the builder's default) assumes
  NO completion masking, `basis="completion"` assumes completion-only masking like this
  notebook's. Since weighting here is informational only (upsampling does the real
  work, and upsampling's own `basis` argument has the same full/completion choice),
  this matters less than it would for a weighted trainer -- but the runbook still has
  you pass `--loss-basis completion` when you build, so `meta.loss_basis` in the file
  matches what this notebook actually does, and the sanity-check cell below WARNS if it
  does not.
- **Train on the assistant completion only.** Every record's `messages` list is
  `[{role: system}, {role: user}, {role: assistant}]` (graphiti's own prompts are
  system+user; the dataset builder appends the target as the assistant turn). Section 3
  uses Unsloth's `train_on_responses_only` to mask every token outside the assistant
  turn from the loss.
- **Start from the BASE model, not the old fine-tune.** `carev01/qwen35-4b-graphrag`
  (the current production student) was distilled from solar-pro4 captures made under
  the OLD, leak-prone extraction instructions and learned that leakage (spec section 1).
  This run starts LoRA from the base `Qwen/Qwen3.5-4B` checkpoint so the new adapter is
  not conditioned on the old model's bad habit.

## 1. Install packages

Standard Unsloth Colab install. `huggingface_hub` is pinned loosely; everything else
follows Unsloth's own recommended install incantation for a T4/L4/A100 Colab runtime.

In [ ]:
%%capture
import importlib.util

if importlib.util.find_spec("unsloth") is None:
    !pip install -q --upgrade pip
    !pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
    !pip install -q --no-deps "trl<0.9.0" peft accelerate bitsandbytes
    !pip install -q --upgrade huggingface_hub datasets
print("packages ready")

## 2. HF token -- Colab secret or environment, never hardcoded

Set a Colab secret named `HF_TOKEN` (the key icon in the left sidebar) *or* export
`HF_TOKEN` in the environment before starting this runtime. This cell never prints the
token -- only whether one was found.

In [ ]:
import os


def get_hf_token() -> str:
    try:
        from google.colab import userdata  # type: ignore
        try:
            tok = userdata.get("HF_TOKEN")
            if tok:
                return tok
        except Exception:
            pass  # secret not set -- fall through to the environment
    except ImportError:
        pass  # not running on Colab
    tok = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")
    if not tok:
        raise RuntimeError(
            "no HF token found -- add a Colab secret named HF_TOKEN, or set the "
            "HF_TOKEN environment variable, before running this cell")
    return tok


HF_TOKEN = get_hf_token()
from huggingface_hub import login
login(token=HF_TOKEN, add_to_git_credential=False)
print("HF token loaded and logged in (value never printed)")

## 3. Configuration

Edit the values in this cell for your run; nothing else below should need changing for
a normal pilot or full run.

In [ ]:
from pathlib import Path

# --- dataset source: "drive" or "hf_dataset" ---------------------------------------
DATA_SOURCE = "hf_dataset"            # or "drive"
HF_DATASET_REPO = "carev01/ft-opus-distill"   # only used when DATA_SOURCE == "hf_dataset"
DRIVE_DATA_DIR = "/content/drive/MyDrive/ft-opus"  # only used when DATA_SOURCE == "drive"

# --- model ----------------------------------------------------------------------------
BASE_MODEL_ID = "Qwen/Qwen3.5-4B"     # the BASE model -- see section 0's rationale
MAX_SEQ_LENGTH = 20480                # spec section 7: context of at least 20k tokens
LOAD_IN_4BIT = True                   # fits an L4 (22.5GB); set False only on a bigger GPU

# --- LoRA -- conservative Unsloth defaults for a 4B model. NOT recovered from a prior
# run: no LoRA rank/lr/epoch/max_seq was found recorded anywhere in this repo for the
# ORIGINAL solar-pro4-distilled fine-tune (checked docs/, scripts/, notebooks/ and the
# HF repo's README text available in this environment) -- see the runbook's "Starting
# hyperparameters" section for the exact grep that came back empty. Treat these as a
# starting point to tune against the eval in section 8, not as recovered truth.
LORA_R = 16
LORA_ALPHA = 16
LORA_DROPOUT = 0.0
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj",
                       "gate_proj", "up_proj", "down_proj"]

# --- training ---------------------------------------------------------------------------
LEARNING_RATE = 2e-4
NUM_TRAIN_EPOCHS = 2
WARMUP_RATIO = 0.03
WEIGHT_DECAY = 0.01
LR_SCHEDULER_TYPE = "cosine"
SEED = 20261017

# per-device batch size x grad accumulation: what fits at MAX_SEQ_LENGTH=20480 with
# gradient checkpointing on. These are STARTING points -- Unsloth's OOM message tells
# you which way to move them:
#   A100 40GB, 4-bit, grad checkpointing: per_device=1, grad_accum=16 (effective batch 16)
#   L4 22.5GB,  4-bit, grad checkpointing: per_device=1, grad_accum=32 (effective batch 32)
PER_DEVICE_TRAIN_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 16       # raise to 32 on an L4
PER_DEVICE_EVAL_BATCH_SIZE = 1

# --- export -----------------------------------------------------------------------------
HF_MODEL_REPO = "carev01/qwen35-4b-graphrag"
HF_MODEL_REVISION = "opus-distill-2026-10"
HF_MODEL_PRIVATE = True

OUT_DIR = Path("/content/outputs/opus-distill")
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("config set")

## 4. Get the dataset

Either mount Drive and point at a folder already holding `train.jsonl` / `val.jsonl` /
`MANIFEST.json` (the output of `scripts/distill/package_dataset.py`, unzipped), or
download the same three files from a private HF **dataset** repo -- whichever the
runbook's packaging step pushed to.

In [ ]:
DATA_DIR = Path("/content/data/ft-opus")
DATA_DIR.mkdir(parents=True, exist_ok=True)

if DATA_SOURCE == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = Path(DRIVE_DATA_DIR)
elif DATA_SOURCE == "hf_dataset":
    from huggingface_hub import hf_hub_download
    for fname in ("train.jsonl", "val.jsonl", "MANIFEST.json", "sample.json"):
        try:
            p = hf_hub_download(repo_id=HF_DATASET_REPO, repo_type="dataset",
                                filename=fname, token=HF_TOKEN)
        except Exception as e:
            if fname == "sample.json":
                print(f"  {fname} not in {HF_DATASET_REPO} (optional, provenance only) -- skipping")
                continue
            raise
        (DATA_DIR / fname).write_bytes(Path(p).read_bytes())
else:
    raise ValueError(f"unknown DATA_SOURCE {DATA_SOURCE!r}")

assert (DATA_DIR / "train.jsonl").exists(), f"no train.jsonl under {DATA_DIR}"
assert (DATA_DIR / "val.jsonl").exists(), f"no val.jsonl under {DATA_DIR}"
print(f"dataset ready at {DATA_DIR}")

## 5. Load train/val and report the realised mix

Reads `meta.loss_weight` / `meta.loss_basis` / `meta.repeat_index` exactly as
`build_extraction_dataset.py` writes them (see its `_record` and `assign_loss_weights`
docstrings) and prints the per-prompt-type share -- purely a sanity check (section 0):
training applies no per-example weight directly, so this cell is how you SEE the mix
the upsampling built, not how you apply it.

In [ ]:
import json
from collections import Counter, defaultdict


def read_jsonl(path: Path) -> list[dict]:
    with path.open(encoding="utf-8") as fh:
        return [json.loads(line) for line in fh if line.strip()]


train_records = read_jsonl(DATA_DIR / "train.jsonl")
val_records = read_jsonl(DATA_DIR / "val.jsonl")
print(f"{len(train_records)} train records, {len(val_records)} val records")

bases = {r["meta"].get("loss_basis") for r in train_records}
if bases - {"completion"}:
    print(f"WARNING: train.jsonl was built with loss_basis={bases!r}, not "
          "'completion' -- this notebook masks prompt tokens (completion-only loss), "
          "so meta.loss_weight's VALUES assumed the wrong masking regime when the "
          "dataset was built. Rebuild with --loss-basis completion if you intend to "
          "act on loss_weight directly; harmless as-is since this notebook relies on "
          "upsampling, not loss_weight, for the mix.")

counts = Counter(r["prompt_name"] for r in train_records)
weights = {r["prompt_name"]: r["meta"].get("loss_weight") for r in train_records}
repeats = defaultdict(int)
for r in train_records:
    repeats[r["prompt_name"]] = max(repeats[r["prompt_name"]],
                                     r["meta"].get("repeat_index", 0) + 1)
total = sum(counts.values())
print(f"{'prompt_name':45s} {'n':>6} {'share':>7} {'loss_weight':>12} {'max_repeat':>11}")
for name, n in sorted(counts.items(), key=lambda kv: -kv[1]):
    print(f"{name:45s} {n:6d} {100*n/total:6.1f}% {weights.get(name):12} {repeats[name]:11d}")

## 6. Load the base model and apply LoRA

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL_ID,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=LOAD_IN_4BIT,
    dtype=None,  # auto: bf16 where supported, else fp16
    token=HF_TOKEN,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=LORA_TARGET_MODULES,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    use_gradient_checkpointing="unsloth",  # long-context checkpointing, needed at 20k tokens
    random_state=SEED,
)
print("base model loaded, LoRA adapter attached")

## 7. Build the chat-formatted dataset, thinking off

Each record's `messages` is `[system, user, assistant]` (section 0). `apply_chat_template`
renders the full three-turn conversation with Qwen's reasoning OFF (`enable_thinking=
False` -- the served production cheap tier always runs thinking off, so training should
match: `scripts/vast/remote_setup.sh`'s `--default-chat-template-kwargs
'{"enable_thinking":false}'`). `packing=False` in the trainer (section 8) keeps every
record's assistant turn individually maskable by `train_on_responses_only`.

In [ ]:
from datasets import Dataset


def to_text(example: dict) -> dict:
    rendered = tokenizer.apply_chat_template(
        example["messages"], tokenize=False, add_generation_prompt=False,
        enable_thinking=False,
    )
    return {"text": rendered}


train_ds = Dataset.from_list(train_records).map(to_text, remove_columns=["messages"])
val_ds = Dataset.from_list(val_records).map(to_text, remove_columns=["messages"])
print(train_ds[0]["text"][:800])

## 8. Train -- completion-only loss, eval on val

`train_on_responses_only` masks every token outside the assistant turn (section 0). The
`instruction_part`/`response_part` markers below are Qwen's ChatML turn openers
(`<|im_start|>user\n` / `<|im_start|>assistant\n`); print one rendered example above and
confirm those two substrings appear verbatim in YOUR tokenizer's template before
trusting the mask -- a mismatch here silently trains on the full sequence instead of
only the completion, the exact defect `drop_overlong`'s docstring warns about in the
opposite direction (truncation instead of masking, same failure mode: silent).

In [ ]:
from trl import SFTConfig, SFTTrainer
from unsloth.chat_templates import train_on_responses_only

sft_config = SFTConfig(
    output_dir=str(OUT_DIR / "checkpoints"),
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,
    lr_scheduler_type=LR_SCHEDULER_TYPE,
    optim="adamw_8bit",
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=3,
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_text_field="text",
    packing=False,  # required for train_on_responses_only's per-example masking
    seed=SEED,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    args=sft_config,
)

trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

train_result = trainer.train()
print(train_result)

In [ ]:
eval_metrics = trainer.evaluate()
print(eval_metrics)

## 9. Export -- merged safetensors + Q8 GGUF, new HF revision

Pushes to `carev01/qwen35-4b-graphrag`, revision `opus-distill-2026-10` -- a NEW
revision, so the current production revision stays untouched and available for
comparison (spec section 7: "the old model stays available for comparison").

In [ ]:
from huggingface_hub import HfApi

api = HfApi(token=HF_TOKEN)
api.create_repo(repo_id=HF_MODEL_REPO, repo_type="model", private=HF_MODEL_PRIVATE,
                exist_ok=True)
try:
    api.create_branch(repo_id=HF_MODEL_REPO, repo_type="model", branch=HF_MODEL_REVISION,
                      exist_ok=True)
except Exception as e:
    print(f"create_branch: {e} (continuing -- push_to_hub_merged below creates it too)")

merged_dir = OUT_DIR / "merged-safetensors"
model.save_pretrained_merged(str(merged_dir), tokenizer, save_method="merged_16bit")

model.push_to_hub_merged(
    HF_MODEL_REPO, tokenizer, save_method="merged_16bit",
    token=HF_TOKEN, revision=HF_MODEL_REVISION, private=HF_MODEL_PRIVATE,
)
print(f"merged safetensors pushed -> {HF_MODEL_REPO}@{HF_MODEL_REVISION}")

In [ ]:
gguf_dir = OUT_DIR / "gguf-q8"
model.save_pretrained_gguf(str(gguf_dir), tokenizer, quantization_method="q8_0")

model.push_to_hub_gguf(
    HF_MODEL_REPO, tokenizer, quantization_method="q8_0",
    token=HF_TOKEN, revision=HF_MODEL_REVISION,
)
print(f"Q8 GGUF pushed -> {HF_MODEL_REPO}@{HF_MODEL_REVISION}")

## 10. Next steps -- outside this notebook

This notebook's job ends here. Hand the revision name (`opus-distill-2026-10`) back to
the runbook for evaluation:

1. `scripts/eval_finetune.py --val <DATA_DIR>/val.jsonl --model <served-name>` against a
   server serving this revision -- validity rate, compared with the teacher.
2. `scripts/extraction_proof.py ... --out report.md` with `CHEAP_LLM_BASE_URL` /
   `CHEAP_LLM_MODEL` pointed at that same server, compared against solar-pro4 and the
   old fine-tune (production's current revision).

See `docs/superpowers/distill-runbook.md` section "Evaluate" for the exact commands and
the pass criteria (spec section 8).